# Training: mcplayhd concept model bundle
_generated with claude based on the other notebooks_

This notebook builds one **model bundle** for the classifier service, from raw replays and hand labels, in one go.
It is the condensed version of the thesis notebooks: features from `02_preprocessing` (section 03), clustering from `03_data-analysis` (section 04b) and voting from `04_cluster_labeling`.

**How to use it**

1. Change what you want in the **settings** cell (paths, window settings, dropped columns, k per mode, ...). Raise `MODEL_VERSION` for every new model.
2. Run all cells. Section 01 reads every replay once and takes a few minutes, everything after it is fast. If only clustering or voting settings changed, re-run from section 02 on, the features stay in memory.
3. The bundle is written to `../models/model_bundle_<MODEL_VERSION>.joblib`. To use it, copy it to `../models/model_bundle.joblib` and restart the server (see `../models/README.md`).

Every step is one small function, so a different idea (another window logic, another clustering algorithm, another voting rule) means replacing one function.
The feature code itself is **imported** from `grl.features`, the same code the server uses, so training and classifying can never drift apart.

**Needs:** the packages from `../requirements.txt` plus jupyter and matplotlib (`requirements.txt` in this folder), the training replays (json list or jsonl) and the hand-label csv.

**Index**

- libraries
- settings
- 01 - features
    - helper functions
    - building dataframe
- 02 - clustering per mode
    - helper functions
    - preparations (scaling & pca)
    - choosing k (optional, slow)
    - k-means
- 03 - voting
    - import hand labels
    - helper functions
    - votes per cluster
- 04 - model bundle
    - export
    - check

## libraries

In [ ]:
import sys
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# the grl package lives one folder up (delivery/grl). python only finds it if that folder is on its search path
sys.path.insert(0, str(Path("..").resolve()))
from grl.features import replay_to_row

## settings

Everything that can be tweaked lives here. Paths are relative to this notebook's folder (`delivery/training`).

In [ ]:
# ---------- paths ----------
DATA_DIR = Path("../../data")
INPUT_DATA = DATA_DIR / "sampled_replays.jsonl"      # training replays: a .jsonl (one replay per line) or a .json list
REPLAY_LABELS_PATH = DATA_DIR / "replay_labels.csv"  # hand labels: idAnonymous, fastBuilderMode, playstyle, suspicious, notes
OUTPUT_DIR = Path("../models")                       # where the bundle gets written

# ---------- model ----------
MODEL_VERSION = "v3_noclicks"  # sent with every api response, raise it for every new model
MODEL_BUNDLE_PATH = OUTPUT_DIR / f"model_bundle_{MODEL_VERSION}.joblib"
RANDOM_SEED = 42               # pca and k-means use it, so the same settings give the same model

# ---------- 01 features ----------
# window settings for grl.features (they get saved in the bundle, so the server builds new replays the same way)
FEATURE_SETTINGS = {
    "CONCEPT_WINDOW_SIZE": 50, # window size in ticks
    "Y_WEIGHT": 0.5,           # weight of the y distance when searching for the best window
    "MAX_CLICKS": 1,           # clicks per tick get capped at this value
    "EYE_HEIGHT": 1.62,        # height of the player's eyes above their position
}
ID_COLUMNS = ["idAnonymous", "fastBuilderMode"]  # columns that describe the replay, not features
DROPPED_COLUMN_PREFIXES = ["clicks_"]            # feature columns the model should NOT see (v3: no clicks), [] keeps all

# ---------- 02 clustering ----------
PCA_VARIANCE = 0.95     # pca keeps as many components as needed to explain this share of the variance
OUTLIER_PERCENTILE = 99 # replays farther from their centre than this percentile are outliers (they don't vote)
RUN_CHOOSING_K = False  # the "choosing k" plots are slow, so they only run when this is True
ELBOW_K_VALUES = range(2, 41, 2)  # k values tried in the "choosing k" plots

# k per mode. modes in the data that are missing here are skipped (the server then can't classify them)
CLUSTER_COUNTS = {
    "EXTRASHORT": 18,
    "SHORT": 20,
    "NORMAL": 20,
    "LONG": 20,
    "ONESTACK": 18,
    "INCLINED": 12,
    "INCLINEDSHORT": 12,
}

# ---------- 03 voting ----------
PLAYSTYLE_SEPARATOR = "_"  # splits a playstyle into its parts, each part is one vote
IGNORED_TECHNIQUE = "ac"   # autoclicker: not listed as technique, its voters don't vote for suspicious

# ---------- 04 bundle ----------
KNN_NEIGHBOURS = 15  # k of the k-NN that puts new replays into a cluster (server side)

## 01 - features

#### helper functions

In [ ]:
def iter_data(path):
    # reads 1 replay at a time, returns a generator of replay dicts
    if path.suffix == ".jsonl":
        # one json per line: only one replay is in memory at a time
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                yield json.loads(line)
    else:
        # one json list (like the thesis data files): has to be loaded as a whole
        with open(path, "r", encoding="utf-8") as f:
            yield from json.load(f)


def build_feature_table(path, feature_settings):
    # runs every replay through grl.features, returns (dataframe with one row per usable replay, number of replays read)
    rows = []
    samples_count = 0
    for item in iter_data(path):
        samples_count += 1
        row = replay_to_row(item, feature_settings) # dict, or None if the replay has no fitting window / no new block
        if row is not None:
            rows.append(row)

    return pd.DataFrame(rows), samples_count

#### building dataframe
Reads every replay once (a few minutes for 70000 replays).

In [ ]:
df_features, samples_count = build_feature_table(INPUT_DATA, FEATURE_SETTINGS)

# sanity checks
print(f"dataframe holds {len(df_features)} out of {samples_count} replays")
print(f"{df_features.shape[1]} columns")
print(df_features["fastBuilderMode"].value_counts())

## 02 - clustering per mode

Every mode gets its own model: StandardScaler → PCA → k-means.
The results go into `clustering_results[mode]`, a dict with the keys the server needs (`ids`, `scaler`, `pca`, `reduced`, `labels`) plus `kmeans`, `nearest_distances` and `is_outlier`.

#### helper functions

In [ ]:
def feature_columns(df_mode):
    # the columns the model gets to see, returns a list of column names (in table order)
    columns = []
    for column in df_mode.columns:
        if column in ID_COLUMNS:
            continue
        if column.startswith(tuple(DROPPED_COLUMN_PREFIXES)): # tuple() because startswith takes one string or a tuple of strings
            continue
        columns.append(column)

    return columns


def scale_and_reduce(df_mode):
    # fits scaler + pca on one mode, returns (scaler, pca, positions in pca space)
    # passing a dataframe (not a numpy array) makes the scaler remember the column names (scaler.feature_names_in_),
    # the server uses them to pick the same columns from new replays
    features = df_mode[feature_columns(df_mode)]
    scaler = StandardScaler()
    scaled = scaler.fit_transform(features)
    pca = PCA(n_components=PCA_VARIANCE, random_state=RANDOM_SEED)
    reduced = pca.fit_transform(scaled)
    return scaler, pca, reduced


def cluster_scores(reduced, k_values):
    # tries every k, returns (inertias, silhouettes) as lists, used for the elbow & silhouette plots
    inertias = []
    silhouettes = []
    for k in k_values:
        km = KMeans(n_clusters=k, random_state=RANDOM_SEED)
        labels = km.fit_predict(reduced)
        inertias.append(km.inertia_)
        silhouettes.append(silhouette_score(reduced, labels))
    return inertias, silhouettes


def cluster_mode(reduced, k):
    # runs k-means on one mode, returns (kmeans, labels, nearest_distances, is_outlier)
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_SEED)
    labels = kmeans.fit_predict(reduced)                     # cluster number of each replay
    nearest_distances = kmeans.transform(reduced).min(axis=1) # distance of each replay to its own (= nearest) centre
    threshold = np.percentile(nearest_distances, OUTLIER_PERCENTILE)
    is_outlier = nearest_distances > threshold                # the farthest replays of the mode
    return kmeans, labels, nearest_distances, is_outlier

#### preparations (scaling & pca)

In [ ]:
clustering_results = {}
for mode in df_features["fastBuilderMode"].unique():
    if mode not in CLUSTER_COUNTS:
        print(f"{mode}: skipped, no k in CLUSTER_COUNTS")
        continue

    df_mode = df_features[df_features["fastBuilderMode"] == mode]
    scaler, pca, reduced = scale_and_reduce(df_mode)
    clustering_results[mode] = {
        "ids": df_mode["idAnonymous"].to_numpy(),
        "scaler": scaler,
        "pca": pca,
        "reduced": reduced,
    }

    print(f"{mode}: {len(df_mode)} replays, {len(scaler.feature_names_in_)} features, {reduced.shape[1]} dimensions after pca")

#### choosing k (optional, slow)
Elbow & silhouette per mode, to pick the values for `CLUSTER_COUNTS`. Takes a long time (one k-means + silhouette per k and mode), so it only runs with `RUN_CHOOSING_K = True`.
Bigger k gives finer clusters but fewer hand-labeled voters per cluster.

In [ ]:
if RUN_CHOOSING_K:
    modes_list = list(clustering_results)
    graph_rows = len(modes_list)

    # squeeze=False keeps axes 2d, even if there is only one mode
    fig, axes = plt.subplots(graph_rows, 2, figsize=(10, 3 * graph_rows), squeeze=False)

    for row, mode in enumerate(modes_list):
        inertias, silhouettes = cluster_scores(clustering_results[mode]["reduced"], ELBOW_K_VALUES)
        replay_count = len(clustering_results[mode]["ids"])

        axes[row, 0].plot(ELBOW_K_VALUES, inertias, marker="o")
        axes[row, 0].set_title(f"{mode} ({replay_count} replays): inertia")
        axes[row, 1].plot(ELBOW_K_VALUES, silhouettes, marker="o")
        axes[row, 1].set_title(f"{mode}: silhouette")

    fig.supxlabel("number of clusters k")
    fig.suptitle("inertia & silhouette per mode")
    fig.tight_layout()
    plt.show()
else:
    print("skipped (RUN_CHOOSING_K = False)")

#### k-means

In [ ]:
for mode, result in clustering_results.items():
    kmeans, labels, nearest_distances, is_outlier = cluster_mode(result["reduced"], CLUSTER_COUNTS[mode])
    result["kmeans"] = kmeans
    result["labels"] = labels
    result["nearest_distances"] = nearest_distances
    result["is_outlier"] = is_outlier

    print(f"{mode}: k={kmeans.n_clusters}, cluster sizes {np.bincount(labels).tolist()}, outliers {is_outlier.sum()}")

## 03 - voting

Every non-outlier hand-labeled replay in a cluster votes:
- `technique_votes`: once for each part of its `playstyle` → `{part: share of voters}`. `IGNORED_TECHNIQUE` (autoclicker) is not listed, but its voters still count in the total.
- `suspicious_votes`: once for its `suspicious` value → `{0-3: count}`. Voters with `IGNORED_TECHNIQUE` are left out here.

A cluster without voters gets two empty dicts ("no information").

#### import hand labels

In [ ]:
df_replay_labels = pd.read_csv(REPLAY_LABELS_PATH)
# drop unlabeled rows from the label file to prevent crashes
df_replay_labels = df_replay_labels.dropna(subset=["playstyle", "suspicious"])
print(f"{len(df_replay_labels)} labeled replays")

#### helper functions

In [ ]:
def counts_dict(values):
    # counts each value, returns a plain {value: count} dict, most common first
    # int() turns numpy numbers into normal python numbers, so the bundle holds plain values
    return {int(key): int(count) for key, count in values.value_counts().items()}


def shares_dict(values, total):
    # counts each value, turns it into a share of total, returns a plain {value: share} dict, most common first
    return {key: round(float(count / total), 2) for key, count in values.value_counts().items()}


def vote_cluster(df_cluster_voters):
    # takes the voters of one cluster, returns {technique_votes, suspicious_votes}
    parts_per_voter = df_cluster_voters["playstyle"].str.split(PLAYSTYLE_SEPARATOR) # "ac_ninja" -> ["ac", "ninja"]
    has_ignored = parts_per_voter.apply(lambda parts: IGNORED_TECHNIQUE in parts)  # True for every autoclicker voter
    technique_parts = parts_per_voter.explode()                                     # one row per part
    technique_parts = technique_parts[technique_parts != IGNORED_TECHNIQUE]
    suspicious_values = df_cluster_voters.loc[~has_ignored, "suspicious"].astype(int)

    voter_count = len(df_cluster_voters) # autoclicker voters stay in the total on purpose
    return {
        "technique_votes": shares_dict(technique_parts, voter_count),
        "suspicious_votes": counts_dict(suspicious_values),
    }


def vote_mode(mode, result):
    # votes every cluster of one mode, returns a list of dicts {mode, cluster, technique_votes, suspicious_votes}
    df_assignments = pd.DataFrame({
        "idAnonymous": result["ids"],
        "cluster": result["labels"],
        "is_outlier": result["is_outlier"],
    })
    # only replays that have a hand label, and no outliers
    df_voters = df_assignments.merge(df_replay_labels[["idAnonymous", "playstyle", "suspicious"]], on="idAnonymous", how="inner")
    df_voters = df_voters[~df_voters["is_outlier"]]

    votes = []
    for cluster in np.unique(result["labels"]): # every cluster, also the ones without voters
        df_cluster_voters = df_voters[df_voters["cluster"] == cluster]
        row = {"mode": mode, "cluster": int(cluster)}
        row.update(vote_cluster(df_cluster_voters))
        votes.append(row)

    return votes

#### votes per cluster

In [ ]:
cluster_votes = []
for mode, result in clustering_results.items():
    cluster_votes.extend(vote_mode(mode, result))

# sanity checks
no_voters = [f"{row['mode']}_{row['cluster']}" for row in cluster_votes if not row["technique_votes"]]
print(f"{len(cluster_votes)} clusters, {len(no_voters)} without voters: {no_voters}")
print(cluster_votes[0])

## 04 - model bundle

One file with everything the server needs. Same format as the bundle the server reads (see `../models/README.md`).

#### export

In [ ]:
bundle = {
    "model_version": MODEL_VERSION,
    "feature_settings": FEATURE_SETTINGS, # the server builds new replays with exactly these settings
    "modes": clustering_results,          # per mode: ids, scaler, pca, reduced, kmeans, labels, nearest_distances, is_outlier
    "cluster_votes": cluster_votes,       # one dict per cluster, votes as real dicts
    "knn_neighbours": KNN_NEIGHBOURS,
}

OUTPUT_DIR.mkdir(exist_ok=True)
joblib.dump(bundle, MODEL_BUNDLE_PATH)
print(f"wrote {len(cluster_votes)} clusters of {len(clustering_results)} modes to {MODEL_BUNDLE_PATH}")

#### check
Loads the written file again, to see that it opens and holds what we expect.

In [ ]:
check_bundle = joblib.load(MODEL_BUNDLE_PATH)
print(f"model_version: {check_bundle['model_version']}")
print(f"feature_settings: {check_bundle['feature_settings']}")
print(f"knn_neighbours: {check_bundle['knn_neighbours']}")
for mode, result in check_bundle["modes"].items():
    print(f"{mode}: {len(result['ids'])} training replays, {result['kmeans'].n_clusters} clusters")